# 🐼 Pandas Part 6: MultiIndex, Reshaping & Pivot Tables Masterclass
**From 1D/2D Basics to Advanced Hierarchical Data & Summarization**

Welcome to Part 6! We are moving beyond simple rows and columns into **Hierarchical Indexing (MultiIndex)**, reshaping data between **Long and Wide formats**, and mastering **Pivot Tables**.

Every concept follows our **12-Step Mastery Framework**:
`Why → Mental Model → Diagram → Code → Mistake → Practice Ladder → Debug → Interview → Real World`


In [1]:
# ==========================================
# 🛠️ SETUP & INDUSTRY MOCK DATA
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print(f"Pandas Version: {pd.__version__}")
print("✅ Environment Ready!")


Pandas Version: 3.0.3
✅ Environment Ready!


# 📘 Module 1: MultiIndex Creation

## 🧠 1. Why it Exists
Suppose we store IPL runs. A normal index only holds one dimension:
`Virat: 973, Rohit: 550`. 
But what if we need data across multiple seasons?
`2023 -> Virat: 973`
`2024 -> Virat: 741`
One index is no longer enough. We need **Hierarchical Indexing**.

## 💡 2. Mental Model
Think of a **Filing Cabinet**:
- **Level 0 (Drawer):** Year (2023, 2024)
- **Level 1 (Folder):** Player (Virat, Rohit)

## 📊 3. Visual Diagram
```text
Normal Index:          MultiIndex (Hierarchical):
Index  | Runs          Level 0 | Level 1 | Runs
Virat  | 973           2023    | Virat   | 973
Rohit  | 550           2023    | Rohit   | 550
                       2024    | Virat   | 741
```

## 💻 4. Code Example
```python
# Method 1: from_tuples (Explicit)
tuples = [('2023', 'Virat'), ('2023', 'Rohit'), ('2024', 'Virat')]
mi1 = pd.MultiIndex.from_tuples(tuples, names=['Year', 'Player'])

# Method 2: from_product (Cartesian Product - Much Faster!)
years = ['2023', '2024']
players = ['Virat', 'Rohit', 'Gill']
mi2 = pd.MultiIndex.from_product([years, players], names=['Year', 'Player'])
```

## ⚠️ 5. Common Mistake
Passing a flat list to `from_tuples` instead of a list of tuples.
```python
pd.MultiIndex.from_tuples(['A', 'B']) # ❌ Fails!
```

## 🎯 6. Practice Ladder (Tasks to Perform)
*   **Level 1 (Easy):** Create a MultiIndex using `from_tuples` for `['Delhi', 'Mumbai']` and `['Q1', 'Q2']`.
*   **Level 2 (Medium):** Create a MultiIndex using `from_product` for `['North', 'South']` and `['Laptop', 'Phone']`.
*   **Level 3 (Hard):** Create a 3-level MultiIndex: `Company -> Department -> Employee`.
*   **Level 4 (Expert):** Generate a 4-level MultiIndex (`Year`, `Quarter`, `Region`, `Product`) using `from_product` and assign random sales data to it.

## 🐛 7. Debugging Challenge
**Why does this fail?**
```python
pd.MultiIndex.from_tuples(['A', 'B'])
```
*Fix it so it creates a valid MultiIndex with one level.*

## 🎤 8. Interview Question
**Q:** What is the difference between `from_tuples()` and `from_product()`?
**A:** `from_tuples` requires you to explicitly define every single row combination. `from_product` automatically generates the Cartesian product (all possible combinations) of the provided lists.

## 🌍 9. Real World Scenario
**E-commerce Categories:** Level 0: `Electronics`, `Clothing`. Level 1: `Laptops`, `Shirts`. This structure allows instant filtering by high-level categories without creating multiple columns.


<details>
<summary>Click for solution</summary>

```
# ==========================================
# ✅ SOLUTIONS (Module 1)
# ==========================================
# Task 1
mi1 = pd.MultiIndex.from_tuples([('Delhi', 'Q1'), ('Delhi', 'Q2'), ('Mumbai', 'Q1'), ('Mumbai', 'Q2')])
print(mi1)

# Task 2
mi2 = pd.MultiIndex.from_product([['North', 'South'], ['Laptop', 'Phone']])
print(mi2)

# Task 3
mi3 = pd.MultiIndex.from_product([['TechCorp'], ['HR', 'IT'], ['Alice', 'Bob']])
print(mi3)

# Task 4
mi4 = pd.MultiIndex.from_product([['2023'], ['Q1', 'Q2'], ['North', 'South'], ['Laptop', 'Phone']])
print(pd.Series(np.random.randint(100, 500, len(mi4)), index=mi4))

# Debugging Fix:
mi_fix = pd.MultiIndex.from_tuples([('A',), ('B',)]) # Wrap in tuples!
print(mi_fix)
```

</details>

In [3]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 1
# ==========================================
# Example Execution
years = ['2023', '2024']
players = ['Virat', 'Rohit', 'Gill']
mi = pd.MultiIndex.from_product([years, players], names=['Year', 'Player'])
runs = [973, 550, 890, 741, 600, 900]
ipl_series = pd.Series(runs, index=mi)
print("MultiIndex Series:\n", ipl_series)

# --- YOUR TURN ---
# Task 1: Create from_tuples for Cities/Quarters
# Task 2: Create from_product for Regions/Products
# Task 3: Create 3-level MultiIndex
# Task 4: 4-level from_product challenge


MultiIndex Series:
 Year  Player
2023  Virat     973
      Rohit     550
      Gill      890
2024  Virat     741
      Rohit     600
      Gill      900
dtype: int64


# 📘 Module 2: Fetching, Slicing & `xs()`

## 🧠 1. Why it Exists
Once data is hierarchical, standard `df['col']` isn't enough. We need to navigate the "folders" (levels) to extract exact data slices.

## 💡 2. Mental Model
Navigating a **Dropdown Menu**: Select Year -> Select Quarter -> Select Region.

## 📊 3. Visual Diagram
```text
ipl_series['2023'] -> Returns all players in 2023
ipl_series.loc[('2023', 'Virat')] -> Returns exact value
ipl_series.xs('Virat', level='Player') -> Returns Virat across ALL years
```

## 💻 4. Code Example
```python
# 1. Partial Indexing (Outer level)
print(ipl_series['2023']) 

# 2. Exact Tuple Lookup
print(ipl_series.loc[('2023', 'Virat')])

# 3. Cross-Section (xs) - Bypasses outer levels!
print(ipl_series.xs('Virat', level='Player'))

# 4. Advanced Slicing with IndexSlice
idx = pd.IndexSlice
print(ipl_series.loc[idx[:, ['Virat', 'Rohit']]]) # All years, specific players
```

## ⚠️ 5. Common Mistake
Trying to access an inner level directly using standard bracket notation.
```python
ipl_series['Virat'] # ❌ KeyError! 'Virat' is Level 1, not Level 0.
```

## 🎯 6. Practice Ladder (Tasks to Perform)
*   **Level 1 (Easy):** Fetch all data for the year '2023'.
*   **Level 2 (Medium):** Fetch the exact runs for ('2024', 'Gill').
*   **Level 3 (Hard):** Use `.xs()` to fetch all data for 'Rohit' regardless of the year.
*   **Level 4 (Expert):** Use `pd.IndexSlice` to fetch data for '2023' AND '2024', but ONLY for 'Virat' and 'Gill'.

## 🐛 7. Debugging Challenge
**Why does this fail?**
```python
unsorted_series = ipl_series.iloc[[2, 0, 1]] # Scrambles order
print(unsorted_series['2023':'2024']) # ❌ KeyError / UnsortedIndexError
```
*Fix it using a built-in Pandas method.*

## 🎤 8. Interview Question
**Q:** What is the dimension (`ndim`) of a MultiIndex Series?
**A:** It is still **1-Dimensional**. Structurally it's a 1D array of values, but the *index* has multiple levels.

## 🌍 9. Real World Scenario
**Financial Reporting:** Using `.xs('Q4', level='Quarter')` to instantly pull Q4 performance across all global regions without filtering the whole DataFrame.


<details>
<summary>Click for Solution</summary>

```
# ==========================================
# ✅ SOLUTIONS (Module 2)
# ==========================================
# Task 1
print(ipl_series['2023'])

# Task 2
print(ipl_series.loc[('2024', 'Gill')])

# Task 3
print(ipl_series.xs('Rohit', level='Player'))

# Task 4
idx = pd.IndexSlice
print(ipl_series.loc[idx[:, ['Virat', 'Gill']]])

# Debugging Fix:
sorted_series = unsorted_series.sort_index() # MUST SORT BEFORE SLICING!
print(sorted_series['2023':'2024'])
```

</details>

In [4]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 2
# ==========================================
# Example Execution
print("2023 Data:\n", ipl_series['2023'])
print("\nVirat's Career:\n", ipl_series.xs('Virat', level='Player'))

# --- YOUR TURN ---
# Task 1: Fetch 2023
# Task 2: Fetch ('2024', 'Gill')
# Task 3: xs() for Rohit
# Task 4: IndexSlice challenge


2023 Data:
 Player
Virat    973
Rohit    550
Gill     890
dtype: int64

Virat's Career:
 Year
2023    973
2024    741
dtype: int64


# 📘 Module 3: Stack & Unstack

## 🧠 1. Why it Exists
Sometimes you need data in a "Tall" format (Series) for memory efficiency, and sometimes you need it in a "Wide" format (DataFrame) for visualization or reporting.

## 💡 2. Mental Model
**Unstack:** Unfolding a map (Rows become Columns).
**Stack:** Folding a map back up (Columns become Rows).

## 📊 3. Visual Diagram
```text
Series (Tall)                 DataFrame (Wide)
2023  Q1  100                 Q1   Q2
      Q2  150      unstack()  ------------
2024  Q1  200  <=========>    2023 | 100 | 150
      Q2  250      stack()    2024 | 200 | 250
```

## 💻 4. Code Example
```python
# Unstack: Innermost index level becomes columns
df_wide = ipl_series.unstack() 
print(df_wide)

# Stack: Columns melt into the innermost index level
s_tall = df_wide.stack()
print(s_tall)

# Unstacking a specific level
df_wide_level0 = ipl_series.unstack(level=0) # Years become columns
```

## ⚠️ 5. Common Mistake
Expecting `unstack()` to work perfectly on data with missing combinations. It will introduce `NaN` values.
```python
# If 2024 Gill is missing, unstack() puts NaN in that cell.
```

## 🎯 6. Practice Ladder (Tasks to Perform)
*   **Level 1 (Easy):** Convert `ipl_series` to a DataFrame where Players are columns.
*   **Level 2 (Medium):** Convert that DataFrame back to a Series.
*   **Level 3 (Hard):** Unstack `level=0` (Years become columns) and observe the difference.
*   **Level 4 (Expert):** Perform `stack() -> unstack() -> stack()` and verify if the data remains identical.



## 🐛 7. Debugging Challenge
**Why does this fail?**
```python
df = pd.DataFrame({'A': [1, 2], 'B': [3, 4]}, index=[('X', 'a'), ('X', 'a')])
df.unstack() # ❌ ValueError: Index contains duplicate entries, cannot reshape
```
*How do you fix the data before unstacking?*

## 🎤 8. Interview Question
**Q:** What is the difference between `stack()` and `melt()`?
**A:** `stack()` moves column *headers* into the index (works on MultiIndex). `melt()` unpivots column *values* into rows (works on standard wide DataFrames).

## 🌍 9. Real World Scenario
**Heatmaps:** Matplotlib/Seaborn require 2D matrices. You must `unstack()` a MultiIndex Series of (City, Month) -> Temperature to plot a heatmap.


<details>
<summary>Click for Solution</summary>

```
# ==========================================
# ✅ SOLUTIONS (Module 3)
# ==========================================
# Task 1
print(ipl_series.unstack())

# Task 2
print(ipl_series.unstack().stack())

# Task 3
print(ipl_series.unstack(level=0))

# Task 4
chain = ipl_series.unstack().stack()
print(chain.equals(ipl_series)) # True

# Debugging Fix:
# The index has duplicates ('X', 'a'). We must aggregate or drop duplicates first.
df_fixed = df.groupby(level=[0,1]).sum() # or .mean()
print(df_fixed.unstack())
```

</details>

In [5]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 3
# ==========================================
# Example Execution
df_wide = ipl_series.unstack()
print("Unstacked:\n", df_wide)

# --- YOUR TURN ---
# Task 1: Unstack to wide
# Task 2: Stack back to tall
# Task 3: Unstack level=0
# Task 4: Stack/Unstack chain


Unstacked:
 Player  Gill  Rohit  Virat
Year                      
2023     890    550    973
2024     900    600    741


# 📘 Module 4: MultiIndex DataFrames & Advanced Ops

## 🧠 1. Why it Exists
DataFrames can have MultiIndex on **Rows**, **Columns**, or **Both**. This allows 4D+ data representation in a 2D object. We also need tools to manipulate these levels (`swaplevel`, `sort_index`, `droplevel`).

## 💡 2. Mental Model
An **Excel Sheet with Merged Cells** for headers.

## 💻 4. Code Example & Advanced Utilities
```python
# Create 4D DataFrame
row_mi = pd.MultiIndex.from_product([['2023', '2024'], ['North', 'South']], names=['Year', 'Region'])
col_mi = pd.MultiIndex.from_product([['Sales', 'Profit'], ['Q1', 'Q2']], names=['Metric', 'Quarter'])
df_4d = pd.DataFrame(np.random.randint(100, 500, (4, 4)), index=row_mi, columns=col_mi)

# --- ADVANCED OPERATIONS ---
# 1. Swap Levels (Flip hierarchy)
df_swapped = df_4d.swaplevel(0, 1, axis=0) # Region becomes outer, Year inner

# 2. Sort Index (Crucial for slicing!)
df_sorted = df_swapped.sort_index()

# 3. Get Level Values (Extract unique categories)
print(df_4d.index.get_level_values('Year').unique())

# 4. Drop Level (Flatten one dimension)
df_dropped = df_4d.droplevel('Quarter', axis=1)

# 5. Reorder Levels
df_reordered = df_4d.reorder_levels(['Quarter', 'Metric'], axis=1)

# 6. Flatten MultiIndex Columns (Make it standard)
df_4d.columns = ['_'.join(col) for col in df_4d.columns.values]
```

## ⚠️ 5. Common Mistake
Using `iloc` for label-based slicing on a MultiIndex.
```python
df_4d.iloc['2023'] # ❌ TypeError: iloc is strictly positional!
```

## 🎯 6. Practice Ladder (Tasks to Perform)
*   **Level 1 (Easy):** Extract all 'Sales' columns from `df_4d`.
*   **Level 2 (Medium):** Swap the row levels so 'Region' is outer, then sort.
*   **Level 3 (Hard):** Use `get_level_values()` to find all unique 'Metrics' in the columns.
*   **Level 4 (Expert):** Flatten the column MultiIndex into single string columns (e.g., `Sales_Q1`).



## 🐛 7. Debugging Challenge
**Why does this fail?**
```python
df_4d.loc['2023':'2024'] # ❌ UnsortedIndexError
```
*Fix it.*

## 🎤 8. Interview Question
**Q:** Are columns really different from the index in Pandas?
**A:** Conceptually, no. Both are `Index` objects. `stack()` literally moves columns into the index, proving they are two sides of the same coin.

## 🌍 9. Real World Scenario
**Flattening for SQL/CSV:** Before exporting a MultiIndex DataFrame to a SQL database or CSV, you *must* flatten the columns using `['_'.join(col)]` so the database accepts standard string headers.


<details>
<summary>Click for Solution</summary>

```
# ==========================================
# ✅ SOLUTIONS (Module 4)
# ==========================================
# Task 1
print(df_4d['Sales'])

# Task 2
print(df_4d.swaplevel(0, 1, axis=0).sort_index())

# Task 3
print(df_4d.columns.get_level_values('Metric').unique())

# Task 4
df_flat = df_4d.copy()
df_flat.columns = ['_'.join(col) for col in df_flat.columns.values]
print(df_flat)

# Debugging Fix:
print(df_4d.sort_index().loc['2023':'2024'])
```

</details>

In [6]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 4
# ==========================================
# Example Execution
row_mi = pd.MultiIndex.from_product([['2023', '2024'], ['North', 'South']], names=['Year', 'Region'])
col_mi = pd.MultiIndex.from_product([['Sales', 'Profit'], ['Q1', 'Q2']], names=['Metric', 'Quarter'])
df_4d = pd.DataFrame(np.random.randint(100, 500, (4, 4)), index=row_mi, columns=col_mi)
print(df_4d)

# --- YOUR TURN ---
# Task 1: Extract Sales
# Task 2: Swap and Sort
# Task 3: get_level_values
# Task 4: Flatten columns


Metric      Sales      Profit     
Quarter        Q1   Q2     Q1   Q2
Year Region                       
2023 North    191  334    299  117
     South    404  394    331  476
2024 North    295  242    365  263
     South    430  403    465  372


# 📘 Module 5: Long vs Wide Data & `melt()`

## 🧠 1. Why it Exists
Machine Learning models and Seaborn plots **hate** Wide data. They need Long (normalized) data. `melt()` is the ultimate unpivoting tool.

## 💡 2. Mental Model
**Wide:** A report card (Columns: Math, Science, English).
**Long:** A database log (Rows: Alice-Math, Alice-Science).

## 📊 3. Visual Diagram
```text
WIDE:                      LONG (Melted):
Name | Math | Sci          Name | Subj  | Score
Alice| 90   | 85    =>     Alice| Math  | 90
Bob  | 80   | 90           Alice| Sci   | 85
                          Bob  | Math  | 80
```

## 💻 4. Code Example
```python
wide_df = pd.DataFrame({
    'Student': ['Alice', 'Bob'],
    'Math': [90, 80],
    'Science': [85, 90]
})

# Melt it!
long_df = pd.melt(
    wide_df, 
    id_vars=['Student'],       # Columns to keep intact
    value_vars=['Math', 'Science'], # Columns to unpivot
    var_name='Subject',        # New name for variable column
    value_name='Score'         # New name for value column
)
```

## ⚠️ 5. Common Mistake
Forgetting `id_vars`, which causes the identifier column to be melted into the values!
```python
pd.melt(wide_df) # ❌ 'Student' becomes part of the Subject/Score mess.
```

## 🎯 6. Practice Ladder (Tasks to Perform)
*   **Level 1 (Easy):** Melt `wide_df` keeping 'Student' as ID.
*   **Level 2 (Medium):** Rename the new columns to 'Subject_Type' and 'Grade_Point'.
*   **Level 3 (Hard):** Melt a DataFrame with 3 ID columns and 4 value columns.
*   **Level 4 (Expert):** Melt the data, then use `groupby('Subject')['Score'].mean()`.


## 🐛 7. Debugging Challenge
**Why is the output weird?**
```python
df = pd.DataFrame({'ID': [1], 'A': [10], 'B': [20]})
pd.melt(df, id_vars='ID', value_vars='A') 
# Output only has 'A'. How do you melt BOTH 'A' and 'B' without typing them?
```

## 🎤 8. Interview Question
**Q:** Why do ML models prefer Long format?
**A:** Long format allows the "Attribute" (e.g., Subject) to be treated as a categorical feature, preventing the creation of sparse, high-dimensional dummy columns for every possible subject.

## 🌍 9. Real World Scenario
**Survey Data:** Converting columns like `Q1_Satisfaction`, `Q2_Satisfaction` into a single `Question` and `Score` column for statistical testing.


<details>
<summary>Click for Solution</summary>

```
# ==========================================
# ✅ SOLUTIONS (Module 5)
# ==========================================
# Task 1
print(pd.melt(wide_df, id_vars=['Student']))

# Task 2
print(pd.melt(wide_df, id_vars=['Student'], var_name='Subject_Type', value_name='Grade_Point'))

# Task 3
df_complex = pd.DataFrame({'ID1':[1], 'ID2':[2], 'ID3':[3], 'V1':[10], 'V2':[20], 'V3':[30], 'V4':[40]})
print(pd.melt(df_complex, id_vars=['ID1', 'ID2', 'ID3']))

# Task 4
print(long_df.groupby('Subject')['Score'].mean())

# Debugging Fix:
# Omit value_vars to melt ALL remaining columns!
print(pd.melt(df, id_vars='ID')) 
```

</details>


In [7]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 5
# ==========================================
# Example Execution
wide_df = pd.DataFrame({'Student': ['Alice', 'Bob'], 'Math': [90, 80], 'Science': [85, 90]})
long_df = pd.melt(wide_df, id_vars=['Student'], var_name='Subject', value_name='Score')
print(long_df)

# --- YOUR TURN ---
# Task 1: Basic melt
# Task 2: Rename var/value
# Task 3: 3 ID cols, 4 value cols
# Task 4: Groupby melted data


  Student  Subject  Score
0   Alice     Math     90
1     Bob     Math     80
2   Alice  Science     85
3     Bob  Science     90


# 📘 Module 6: Pivot Tables

## 🧠 1. Why it Exists
To create multidimensional summary tables (Cross-tabulations). It's the Pandas equivalent of Excel Pivot Tables.

## 💡 2. Mental Model
A **Rubik's Cube**: Rows (Index), Columns, and Values (Aggregated inside).

## 💻 4. Code Example & Advanced Parameters
```python
sales_df = pd.DataFrame({
    'Region': ['North', 'North', 'South', 'South'],
    'Product': ['Laptop', 'Phone', 'Laptop', 'Phone'],
    'Revenue': [1000, 500, 800, 600],
    'Units': [10, 5, 8, 6]
})

# Basic Pivot
pivot1 = pd.pivot_table(sales_df, index='Region', columns='Product', values='Revenue', aggfunc='sum')

# Advanced: Multiple Aggs, Margins, Fill Value
pivot2 = pd.pivot_table(
    sales_df, 
    index='Region', 
    columns='Product', 
    values=['Revenue', 'Units'],
    aggfunc={'Revenue': 'sum', 'Units': 'mean'}, # Different agg per column
    margins=True,       # Adds 'All' row/col for grand totals
    fill_value=0,       # Replaces NaN with 0
    observed=True       # Crucial for Categorical data!
)
```

## ⚠️ 5. Common Mistake
Using `pivot()` instead of `pivot_table()` when duplicates exist.
```python
sales_df.pivot(index='Region', columns='Product', values='Revenue') 
# ❌ ValueError: Index contains duplicate entries, cannot reshape
```

## 🎯 6. Practice Ladder (Tasks to Perform)
*   **Level 1 (Easy):** Create a pivot table of mean Revenue by Region and Product.
*   **Level 2 (Medium):** Add `margins=True` to see grand totals.
*   **Level 3 (Hard):** Apply different `aggfunc` to Revenue (sum) and Units (mean).
*   **Level 4 (Expert):** Create a pivot table with a MultiIndex row (`['Region', 'Product']`) and no columns.


## 🐛 7. Debugging Challenge
**Why does this fail?**
```python
df = pd.DataFrame({'A': ['x', 'y'], 'B': ['a', 'b'], 'C': [1, 2]})
pd.pivot_table(df, index='A', columns='B', values='D') # ❌ KeyError: 'D'
```
*How do you pivot ALL numeric columns without specifying `values`?*

## 🎤 8. Interview Question
**Q:** What is the difference between `pivot_table()` and `groupby()`?
**A:** `groupby()` returns a flattened Series/DataFrame with a MultiIndex. `pivot_table()` returns a 2D rectangular DataFrame (cross-tabulation) and is generally more readable for business reporting.

## 🌍 9. Real World Scenario
**Executive Dashboards:** Summarizing 100,000 transaction rows into a neat 5x5 matrix showing Total Revenue by Region (Rows) and Product Category (Columns) with Grand Totals on the edges.


<details>
<summary>Click for Solution</summary>

```
# ==========================================
# ✅ SOLUTIONS (Module 6)
# ==========================================
# Task 1
print(pd.pivot_table(sales_df, index='Region', columns='Product', values='Revenue', aggfunc='mean'))

# Task 2
print(pd.pivot_table(sales_df, index='Region', columns='Product', values='Revenue', aggfunc='sum', margins=True))

# Task 3
print(pd.pivot_table(sales_df, index='Region', columns='Product', values=['Revenue', 'Units'], aggfunc={'Revenue': 'sum', 'Units': 'mean'}))

# Task 4
print(pd.pivot_table(sales_df, index=['Region', 'Product'], values='Revenue', aggfunc='sum'))

# Debugging Fix:
# Omit the 'values' parameter to pivot all remaining numeric columns!
print(pd.pivot_table(df, index='A', columns='B', aggfunc='sum'))
```

</details>


In [8]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 6
# ==========================================
# Example Execution
sales_df = pd.DataFrame({
    'Region': ['North', 'North', 'South', 'South'],
    'Product': ['Laptop', 'Phone', 'Laptop', 'Phone'],
    'Revenue': [1000, 500, 800, 600],
    'Units': [10, 5, 8, 6]
})

pivot = pd.pivot_table(sales_df, index='Region', columns='Product', values='Revenue', aggfunc='sum', fill_value=0)
print(pivot)

# --- YOUR TURN ---
# Task 1: Mean Revenue pivot
# Task 2: Add margins
# Task 3: Multiple aggfuncs
# Task 4: MultiIndex row pivot


Product  Laptop  Phone
Region                
North      1000    500
South       800    600


# 🎉 Congratulations!
You have mastered **Pandas Part 6**. 
You can now navigate hierarchical data, reshape formats for ML/Viz, and build complex business summaries.

### 🚀 Next Steps
You are now ready for **Pandas Part 7: Time Series Analysis** – handling dates, resampling, rolling windows, and time-based indexing!
